# Embeddings Playground — coding workshop (beginner)

**Goes with the Session 1 lesson, Step 2 · Embeddings, after the screen “What we keep: the
embedding table.”** Expected time: ~25 minutes, individual. No GPU needed; the training
runs take about three minutes in total.

In the lesson, skip-gram learned vectors for 8 words from a 12-word corpus, and with 2
dimensions the similar words found each other sooner than with 92. Here you train the same
model, word2vec's skip-gram, on 2 million words of Galician Wikipedia:

- **Part A:** train with 2 to 300 dimensions, and measure time, energy and quality.
- **Part B:** train on less text, and see which words suffer: the lesson's claim that rare
  words get poorly placed vectors.
- **Part C:** test a multilingual model on Galician words that differ from Spanish.

## How this workshop works

This is the coding version of the notebook, **beginner level**. Some code cells have gaps
marked `___`. An exercise above each such cell says what to write. This version takes
longer than the expected time given above.

1. Read the exercise. Open the hints if you need them.
2. Replace every `___` in the cell with your code, then run the cell.
3. Run the check cell right below it. It prints `✓ Exercise N passed`, or an error
   message that says what to look at.
4. If you are stuck, the solution is below the check cell. Open it after you have tried.

A cell that still contains `___` stops with `NameError: name '___' is not defined`.
Later cells use your code, so finish each exercise before moving on.

## Part A — Train skip-gram on Galician

The text is the Galician sample from Notebook 01b: about 2 million words from random
articles of Galician Wikipedia (CC BY-SA 4.0).

In [ ]:
!pip install -q gensim codecarbon

import gzip
import re
import time
import urllib.request
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

DATA_URL = "https://olzama.github.io/teaching/GreenAI/corpora"
WORD = re.compile(r"[^\W\d_]+")      # a word is a run of letters

urllib.request.urlretrieve(f"{DATA_URL}/wiki_gl_2M.txt.gz", "wiki_gl_2M.txt.gz")
with gzip.open("wiki_gl_2M.txt.gz", "rt", encoding="utf-8") as f:
    articles = [WORD.findall(line.lower()) for line in f]
word_counts = Counter(w for a in articles for w in a)
print(f"{len(articles):,} articles, {sum(word_counts.values()):,} words, {len(word_counts):,} different words")

### How to tell whether the vectors are good

Five groups of Galician words. A good embedding puts each word close to the other words of
its group. For every word, take its 3 nearest neighbors among these 44 words; the *group
score* is the share of those neighbors that belong to the same group. 1.0 is perfect.

In [ ]:
GROUPS = {
    "months": "xaneiro febreiro marzo abril maio xuño xullo agosto setembro outubro novembro decembro".split(),
    "days": "luns martes mércores xoves venres sábado domingo".split(),
    "numbers": "dous tres catro cinco seis sete oito nove dez".split(),
    "colors": "vermello azul verde amarelo branco negro".split(),
    "family": "nai pai irmán irmá fillo filla avó avoa tío tía".split(),
}
GROUP_OF = {w: g for g, ws in GROUPS.items() for w in ws}

for g, ws in GROUPS.items():
    print(f"{g:>8}: " + ", ".join(f"{w} {word_counts[w]}" for w in ws))

### ✏️ Exercise 1 — The group score

Complete `group_scores`. For each test word, find its `k` nearest test words by cosine
similarity, and return the share of them in the same group.

The vectors in `V` are already scaled to length 1, so `S = V @ V.T` holds every cosine
similarity, and the diagonal is set to −∞ so a word is never its own neighbor.

<details><summary><b>Hints</b> (click to open)</summary>

- `np.argsort(x)` sorts from smallest to largest: sort `-S[i]` to get the most similar first.
- `GROUP_OF[words[j]] == GROUP_OF[w]` is True when neighbor j is in w's group; the mean of booleans is the share.

</details>

In [ ]:
def group_scores(wv, k=3):
    """Group score of every test word that has a vector: {word: share of its k nearest test words in its group}."""
    words = [w for w in GROUP_OF if w in wv]
    V = np.array([wv[w] for w in words])
    V = V / np.linalg.norm(V, axis=1, keepdims=True)   # unit length: dot product = cosine
    S = V @ V.T
    np.fill_diagonal(S, -np.inf)                         # a word is not its own neighbor
    scores = {}
    for i, w in enumerate(words):
        nearest = np.argsort(___)[:k]   # indices, most similar first
        scores[w] = np.mean([___ for j in nearest])
    return scores

In [ ]:
# Check for Exercise 1. Run this cell as is.
def test_perfect():
    """two well-separated groups score 1.0"""
    wv = {w: np.array([1.0, 0.05 * i]) for i, w in enumerate(GROUPS["months"][:4])}
    wv.update({w: np.array([0.05 * i, 1.0]) for i, w in enumerate(GROUPS["days"][:4])})
    s = group_scores(wv)
    assert set(s) == set(wv) and all(v == 1.0 for v in s.values())

def test_mixed():
    """a word among the other group scores 0"""
    wv = {"xaneiro": np.array([1.0, 0.0]), "febreiro": np.array([0.95, 0.31]), "luns": np.array([0.0, 1.0])}
    s = group_scores(wv, k=1)
    assert s == {"xaneiro": 1.0, "febreiro": 1.0, "luns": 0.0}, s

def test_missing_words():
    """words without a vector are left out"""
    s = group_scores({"tía": np.array([1.0, 0.0]), "tío": np.array([0.9, 0.1])}, k=1)
    assert set(s) == {"tía", "tío"}

_failed = 0
for _test in [test_perfect, test_mixed, test_missing_words]:
    try:
        _test()
        print("✓", _test.__doc__)
    except Exception as _e:
        _failed += 1
        print("✗", _test.__doc__)
        print("   ", _e if isinstance(_e, AssertionError) else f"{type(_e).__name__}: {_e}")
assert _failed == 0, f"{_failed} test(s) failed: see the ✗ lines above"
print("✓ Exercise 1 passed")

<details><summary><b>Solution to Exercise 1</b> (open it after you have tried)</summary>

```python
def group_scores(wv, k=3):
    """Group score of every test word that has a vector: {word: share of its k nearest test words in its group}."""
    words = [w for w in GROUP_OF if w in wv]
    V = np.array([wv[w] for w in words])
    V = V / np.linalg.norm(V, axis=1, keepdims=True)   # unit length: dot product = cosine
    S = V @ V.T
    np.fill_diagonal(S, -np.inf)                         # a word is not its own neighbor
    scores = {}
    for i, w in enumerate(words):
        nearest = np.argsort(-S[i])[:k]
        scores[w] = np.mean([GROUP_OF[words[j]] == GROUP_OF[w] for j in nearest])
    return scores
```

</details>

### Train with 2 to 300 dimensions

Skip-gram with a window of 5 words on each side, words that occur at least 5 times, 5
passes over the text. CodeCarbon measures each run.

### ✏️ Exercise 2 — Train skip-gram

Complete the call to `Word2Vec`: skip-gram (`sg=1`; `sg=0` is the other word2vec
model, CBOW), with `dims` dimensions (`vector_size`).

<details><summary><b>Hints</b> (click to open)</summary>

- `vector_size=dims`, `sg=1`.

</details>

In [ ]:
from gensim.models import Word2Vec
from codecarbon import EmissionsTracker


def get_tracker_energy_kwh(tracker):
    """Total energy of a stopped tracker, in kWh (the attribute moved between CodeCarbon versions)."""
    for attr_path in ("final_emissions_data.energy_consumed", "_total_energy.kWh"):
        obj = tracker
        try:
            for part in attr_path.split("."):
                obj = getattr(obj, part)
            return float(obj)
        except Exception:
            continue
    return None


def train(texts, dims):
    """Train skip-gram (sg=1) and measure it: returns (model, seconds, Wh)."""
    tracker = EmissionsTracker(measure_power_secs=1, save_to_file=False, log_level="error")
    tracker.start()
    t0 = time.perf_counter()
    model = Word2Vec(texts, vector_size=___, window=5, min_count=5, sg=___,
                     negative=5, epochs=5, workers=2, seed=0)
    seconds = time.perf_counter() - t0
    tracker.stop()
    return model, seconds, get_tracker_energy_kwh(tracker) * 1000


runs = []
for dims in [2, 10, 50, 100, 300]:
    model, seconds, wh = train(articles, dims)
    score = np.mean(list(group_scores(model.wv).values()))
    runs.append((dims, seconds, wh, score))
    print(f"{dims:>3} dimensions: {seconds:5.1f} s, {wh:.3f} Wh, group score {score:.2f}")

In [ ]:
# Check for Exercise 2. Run this cell as is.
assert [r[0] for r in runs] == [2, 10, 50, 100, 300]
assert model.sg == 1, f"the model has sg={model.sg}: pass sg=1 to Word2Vec for skip-gram (the default, sg=0, is CBOW)"
assert model.wv.vector_size == 300, "the last run has 300 dimensions"
assert runs[2][3] > runs[0][3], "50 dimensions should beat 2"
print("✓ Exercise 2 passed")

<details><summary><b>Solution to Exercise 2</b> (open it after you have tried)</summary>

```python
from gensim.models import Word2Vec
from codecarbon import EmissionsTracker


def get_tracker_energy_kwh(tracker):
    """Total energy of a stopped tracker, in kWh (the attribute moved between CodeCarbon versions)."""
    for attr_path in ("final_emissions_data.energy_consumed", "_total_energy.kWh"):
        obj = tracker
        try:
            for part in attr_path.split("."):
                obj = getattr(obj, part)
            return float(obj)
        except Exception:
            continue
    return None


def train(texts, dims):
    """Train skip-gram (sg=1) and measure it: returns (model, seconds, Wh)."""
    tracker = EmissionsTracker(measure_power_secs=1, save_to_file=False, log_level="error")
    tracker.start()
    t0 = time.perf_counter()
    model = Word2Vec(texts, vector_size=dims, window=5, min_count=5, sg=1,
                     negative=5, epochs=5, workers=2, seed=0)
    seconds = time.perf_counter() - t0
    tracker.stop()
    return model, seconds, get_tracker_energy_kwh(tracker) * 1000


runs = []
for dims in [2, 10, 50, 100, 300]:
    model, seconds, wh = train(articles, dims)
    score = np.mean(list(group_scores(model.wv).values()))
    runs.append((dims, seconds, wh, score))
    print(f"{dims:>3} dimensions: {seconds:5.1f} s, {wh:.3f} Wh, group score {score:.2f}")
```

</details>

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot([r[2] for r in runs], [r[3] for r in runs], "o-")
for dims, s, wh, score in runs:
    plt.annotate(f"{dims} dims", (wh, score), textcoords="offset points", xytext=(6, -10))
plt.xlabel("energy of training, Wh")
plt.ylabel("group score")
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.show()

**Reflection:** How does the group score change from 2 to 300 dimensions? And the time and
energy? Which number of dimensions would you choose for this corpus, and why? Why might 2
dimensions be too few for 44 words in five groups?

## Part B — Less text

The lesson said: a rare word takes part in few training pairs, gets few nudges, and ends up
poorly placed. Train with 50 dimensions on the first 100,000 words, then more, and look at
each group separately. The word counts above show which groups are rare.

In [ ]:
def first_words(arts, n_words):
    out, n = [], 0
    for a in arts:
        if n >= n_words:
            break
        out.append(a[:n_words - n])
        n += len(out[-1])
    return out


print(f"{'words':>10} | {'test words with a vector':>24} | " + " | ".join(f"{g:>7}" for g in GROUPS))
for n_words in [100_000, 300_000, 1_000_000, 2_000_000]:
    model, seconds, wh = train(first_words(articles, n_words), 50)
    scores = group_scores(model.wv)
    per_group = [np.mean([scores[w] for w in ws if w in scores]) if any(w in scores for w in ws) else float("nan")
                 for ws in GROUPS.values()]
    print(f"{n_words:>10,} | {len(scores):>24} | " + " | ".join(f"{s:>7.2f}" for s in per_group))

**Reflection:** Which group is good already with 100,000 words, and which is worst? Compare with
the word counts printed in Part A. What does that say about rare words, and about a language
with little text online?

### Nearest neighbors and a map

The model trained on all 2 million words, with 50 dimensions. Its nearest neighbors come
from the whole vocabulary, not only the 44 test words.

In [ ]:
model, _, _ = train(articles, 50)
for w in ["xaneiro", "verde", "tía"]:
    print(f"{w}: " + ", ".join(f"{n} {s:.2f}" for n, s in model.wv.most_similar(w, topn=6)))

from sklearn.decomposition import PCA

words = [w for w in GROUP_OF if w in model.wv]
coords = PCA(n_components=2).fit_transform(np.array([model.wv[w] for w in words]))
plt.figure(figsize=(7, 6))
for g in GROUPS:
    idx = [i for i, w in enumerate(words) if GROUP_OF[w] == g]
    plt.scatter(coords[idx, 0], coords[idx, 1], label=g)
for (x, y), w in zip(coords, words):
    plt.annotate(w, (x, y), textcoords="offset points", xytext=(4, 4), fontsize=8)
plt.title("50-dimensional Galician skip-gram vectors, projected to 2-D")
plt.legend()
plt.show()

## Part C — A multilingual model and Galician words

`paraphrase-multilingual-MiniLM-L12-v2` is trained on translated sentence pairs in about 50
languages, so translations land close together. Galician is on its model card's list of
languages. Can it match Galician words with their Spanish and English translations?

The test: for each Galician word, find the nearest word among all the Spanish words, and
among all the English words. The first 13 Galician words are spelled differently from
Spanish; the last 4 are spelled the same.

### ✏️ Exercise 3 — Nearest translation

Each row of `S_es` holds the cosine similarities of one Galician word to all the
Spanish words (`S_en`: to the English words). Find the index of the nearest one.

<details><summary><b>Hints</b> (click to open)</summary>

- `S_es[i].argmax()` is the column with the largest similarity in row i.

</details>

In [ ]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

multi_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

WORDS = [("can", "perro", "dog"), ("fiestra", "ventana", "window"), ("cadeira", "silla", "chair"),
         ("neno", "niño", "boy"), ("irmá", "hermana", "sister"), ("avoa", "abuela", "grandmother"),
         ("mazá", "manzana", "apple"), ("leite", "leche", "milk"), ("noite", "noche", "night"),
         ("fillo", "hijo", "son"), ("cidade", "ciudad", "city"), ("auga", "agua", "water"),
         ("lingua", "lengua", "language"),
         ("tía", "tía", "aunt"), ("mesa", "mesa", "table"), ("casa", "casa", "house"), ("mar", "mar", "sea")]

gl, es, en = (multi_model.encode([t[i] for t in WORDS], normalize_embeddings=True) for i in range(3))
S_es, S_en = gl @ es.T, gl @ en.T          # cosine similarities: row = Galician word

right = {"es": 0, "en": 0}
print(f"{'Galician':>9} | {'nearest Spanish':>20} | {'nearest English':>20}")
for i, (g, s, e) in enumerate(WORDS):
    js, je = int(___), int(___)
    right["es"] += js == i
    right["en"] += je == i
    mark = lambda ok: "✓" if ok else "✗"
    print(f"{g:>9} | {WORDS[js][1]:>13} {S_es[i, js]:.2f} {mark(js == i)} | {WORDS[je][2]:>13} {S_en[i, je]:.2f} {mark(je == i)}")
print(f"\nright: Spanish {right['es']} of {len(WORDS)}, English {right['en']} of {len(WORDS)}")

In [ ]:
# Check for Exercise 3. Run this cell as is.
assert right["es"] == sum(int(S_es[i].argmax()) == i for i in range(len(WORDS)))
assert right["en"] == sum(int(S_en[i].argmax()) == i for i in range(len(WORDS)))
assert right["es"] >= 4, "the 4 words spelled like Spanish match themselves"
print("✓ Exercise 3 passed")

<details><summary><b>Solution to Exercise 3</b> (open it after you have tried)</summary>

```python
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

multi_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

WORDS = [("can", "perro", "dog"), ("fiestra", "ventana", "window"), ("cadeira", "silla", "chair"),
         ("neno", "niño", "boy"), ("irmá", "hermana", "sister"), ("avoa", "abuela", "grandmother"),
         ("mazá", "manzana", "apple"), ("leite", "leche", "milk"), ("noite", "noche", "night"),
         ("fillo", "hijo", "son"), ("cidade", "ciudad", "city"), ("auga", "agua", "water"),
         ("lingua", "lengua", "language"),
         ("tía", "tía", "aunt"), ("mesa", "mesa", "table"), ("casa", "casa", "house"), ("mar", "mar", "sea")]

gl, es, en = (multi_model.encode([t[i] for t in WORDS], normalize_embeddings=True) for i in range(3))
S_es, S_en = gl @ es.T, gl @ en.T          # cosine similarities: row = Galician word

right = {"es": 0, "en": 0}
print(f"{'Galician':>9} | {'nearest Spanish':>20} | {'nearest English':>20}")
for i, (g, s, e) in enumerate(WORDS):
    js, je = int(S_es[i].argmax()), int(S_en[i].argmax())
    right["es"] += js == i
    right["en"] += je == i
    mark = lambda ok: "✓" if ok else "✗"
    print(f"{g:>9} | {WORDS[js][1]:>13} {S_es[i, js]:.2f} {mark(js == i)} | {WORDS[je][2]:>13} {S_en[i, je]:.2f} {mark(je == i)}")
print(f"\nright: Spanish {right['es']} of {len(WORDS)}, English {right['en']} of {len(WORDS)}")
```

</details>

**Reflection:** How many Galician words did the model match with Spanish, and with English?
Which words did it get wrong? What do they have in common? Why do the last 4 words get a
similarity of exactly 1.00 with Spanish? What does this tell you about a model that saw
little Galician?

## Code literacy check

Before you move on, pick at least two code cells above that you actually ran and read
today, and add `#` comments to them explaining, in your own words, what each meaningful
line or block does and *why* it's there, in language a classmate who hasn't seen the code
could follow.

Do this part on your own, after class:

1. Paste your commented code into an LLM chat (ChatGPT, Claude, Gemini — whichever you
   have access to).
2. Ask it something like: *"Here is code I commented myself, explaining what I think it
   does. Based only on my comments, how well do you think I understood this code? What
   did I get right, what did I miss, and what's still fuzzy?"*
3. Read what it says. Where it disagrees with your comment, go check the code again —
   who's right?

This is a self-check on your own understanding: a way to catch, in your own words, where
your mental model of the code breaks down against a second reader.